In [ ]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

from datasets import load_dataset
from matplotlib import pyplot as plt

import random
import json
from PIL import Image
import pandas as pd
from tqdm import tqdm

import numpy as np

import torch
import torchvision

from utils import *

def choose_device() -> str:
    if torch.cuda.is_available():
        return "cuda:0"
    if hasattr(torch.backends, "mps"):
        if torch.backends.mps.is_available():
            return "mps"
    return "cpu"

device = torch.device(choose_device())
print(torch.cuda.get_device_name(device))

# fix seed for reproducibility
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed) # multi-GPU

In [ ]:
imagenet_transform = torchvision.transforms.Compose(
    [
        SquareCropAndResize(224),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        ),
    ]
)

inverse_transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Normalize(
            mean=[0.0, 0.0, 0.0], std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
        ),
        torchvision.transforms.Normalize(
            mean=[-0.485, -0.456, -0.406], std=[1.0, 1.0, 1.0]
        ),
    ]
)

with open("imagenet_class_index.json") as f:
    imagenet_labels = list(json.load(f).values())


In [ ]:
from utils import *
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

resnet = torchvision.models.resnet50(weights="ResNet50_Weights.IMAGENET1K_V1")
resnet.eval().to("cuda")

# Select the LayerCAM XAI method
xai_method_name, method = ("LayerCAM", torchcam_LayerCAM(resnet, resnet.layer3, imagenet_labels, None))

# Load the image from the path
img_path = 'picture.jpeg'
img = Image.open(img_path).convert('RGB')  # Convert to RGB
img = SquareCropAndResize(224)(img)  # Assuming SquareCropAndResize is defined in utils

# Apply transformations to the image for model input
imagenet_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
img_tensor = imagenet_transform(img).unsqueeze(0).to('cuda')

# Generate the saliency map
heatmap, pred_label = method(img_tensor)
heatmap = heatmap[0].cpu().numpy()  # Do not flip heatmap vertically

# Apply threshold to dim values below 0.3
threshold = 0.3
dim_factor = 0.8  # Factor to dim the heatmap below threshold
dimmed_heatmap = np.where(heatmap >= threshold, heatmap, heatmap * dim_factor)

# Upsample heatmap to match the image size
heatmap_resized = np.kron(dimmed_heatmap, np.ones((16, 16)))  # Adjust factor based on original heatmap size

# Display the original image and the overlay
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img)
ax.axis('off')
ax.set_title(f"{xai_method_name} - Predicted Label: {pred_label}")

# Overlay the heatmap on the original image
cmap_overlay = ax.imshow(heatmap_resized, cmap='Reds', alpha=0.5, extent=[0, 224, 224, 0])  # Use red colormap
fig.colorbar(plt.cm.ScalarMappable(cmap='Reds', norm=plt.Normalize(vmin=heatmap_resized.min(), vmax=heatmap_resized.max())), ax=ax)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import torch
torch.cuda.empty_cache()
import matplotlib.pyplot as plt
import cv2

In [ ]:
def show_anns(anns):
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)

    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:,:,3] = 0
    for ann in sorted_anns:
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.35]])
        img[m] = color_mask
    ax.imshow(img)

In [ ]:
image = cv2.imread('picture.jpeg')
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [ ]:
import sys
sys.path.append("..")
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

sam_checkpoint = "sam_vit_b_01ec64.pth"
model_type = "vit_b"

torch.cuda.empty_cache()
device = "cuda"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

mask_generator = SamAutomaticMaskGenerator(sam)

In [ ]:
masks = mask_generator.generate(image)

In [ ]:
print(len(masks))
print(masks[0].keys())